In [18]:
import numpy as np
import h5py
import time
%load_ext autoreload
import os
%autoreload 2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import mpl_toolkits.mplot3d.axes3d as p3
import matplotlib.animation as animation
from ipywidgets import Video
%matplotlib widget


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# load the tracking results

tracksPath = '/home/liam/fish_analysis/nugget_dump/tracking_20201119_batch1/FishTank20200130_153857_results.h5'

# load image coordinates data
with h5py.File(tracksPath, 'r') as hf:
    tracks_3D = hf['tracks_3D_smooth'][:]
    
tracks_3D.shape

(501943, 2, 3, 3)

In [15]:
# Functions 


def animate_scatters(iteration, data, scatters, lines, scatter_projections, lines_projections, text, all_timestamps):
    """
    Update the data held by the scatter plot and therefore animates it.

    Args:
        iteration (int): Current iteration of the animation
        data (list): List of the data positions at each iteration.
        scatters (list): List of all the scatters (One per element)

    Returns:
        list: List of scatters (One per element) with new coordinates
    """
    #if np.mod(iteration, 100) == 0:
        #print(iteration)
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
 
    # update the scatters
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            scatters[fishIdx][bpIdx]._offsets3d = (data[iteration,fishIdx,bpIdx,0:1],
                                                   data[iteration,fishIdx,bpIdx,1:2],
                                                   data[iteration,fishIdx,bpIdx,2:])
            
    # update the lines
    for fishIdx,line in enumerate(lines):
        line.set_data(data[iteration, fishIdx, :, :2].swapaxes(0,1))
        line.set_3d_properties(data[iteration, fishIdx, :, 2])
        
        
    # update the text
    text.set_text(all_timestamps[iteration])
        
    # -------------- #
    for projectionIdx in range(3):
        
        # XZ => Y=0
        if projectionIdx == 0:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 1] = 0
        # XY => Z=0
        elif projectionIdx == 1:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 2] = 0
        # YZ => X=0
        elif projectionIdx == 2:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 0] = 0
            
        proj_scatters = scatter_projections[projectionIdx]
        proj_lines = lines_projections[projectionIdx]
        
        # update the scatters
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                proj_scatters[fishIdx][bpIdx]._offsets3d = (projectionData[fishIdx,bpIdx,0:1],
                                                            projectionData[fishIdx,bpIdx,1:2],
                                                            projectionData[fishIdx,bpIdx,2:])
        # update the lines
        for fishIdx,line in enumerate(proj_lines):
            line.set_data(projectionData[fishIdx, :, :2].swapaxes(0,1))
            line.set_3d_properties(projectionData[fishIdx, :, 2])

    return




def make_movie(data, f0, fE, savePath='/home/liam/temp/testanimation.mp4'):
    """
    Creates the 3D figure and animates it with the input data.

    Args:
        data (list): List of the data positions at each iteration.
        save (bool): Whether to save the recording of the animation. (Default to False).
    """
    data = tracks_3D[f0:fE]
    
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
    fish_colors = ['red', 'blue']

    # Attaching 3D axis to the figure
    fig = plt.figure()
    fig.tight_layout()
    ax = p3.Axes3D(fig, [0.,0.,1.,1.])

    # Initialize scatters (list over fish, list over bps)
    symbols = ['o', 's', 'x']
    sizes=12
    
    # Main scatters
    scatters = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(data[0,fishIdx,bpIdx,0:1], data[0,fishIdx,bpIdx,1:2], data[0,fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx])) 
        scatters.append(fish_scatters)
        
    # Main lines
    lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        line = ax.plot(data[0, fishIdx, :, 0], data[0, fishIdx, :, 1], data[0, fishIdx, :, 2], c=col)[0]
        lines.append(line)
        
    # ----------------
    # projections
    symbols = ['x', 'o', 's']
    sizes=5

    scatter_projections = []
    lines_projections = []
    for projectionIdx in range(3):
        proj_scatters = []
        proj_lines = []
        for fishIdx in range(numFish):
            col = fish_colors[fishIdx]
            # scatter plots
            fish_scatters = []
            for bpIdx in range(numBodyPoints):
                #print(symbols[bpIdx])
                fish_scatters.append(ax.scatter(data[0,fishIdx,bpIdx,0:1], data[0,fishIdx,bpIdx,1:2], data[0,fishIdx,bpIdx,2:], 
                                                c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
            # lines
            line = ax.plot(data[0, fishIdx, :, 0], data[0, fishIdx, :, 1], data[0, fishIdx, :, 2], c=col, alpha=0.1)[0]
            proj_lines.append(line)
            proj_scatters.append(fish_scatters)
        
        scatter_projections.append(proj_scatters)
        lines_projections.append(proj_lines)
    # ----------------
    
    # the text
    all_timestamps = ['frame: '+ str(i).zfill(7) for i in range(f0, fE)]
    time_text = ax.text(14, 0, 43,'', fontsize=15)

    # Setting the axes properties
    ax.set_xlim3d([0, 40])
    ax.set_xlabel('X')

    ax.set_ylim3d([0, 40])
    ax.set_ylabel('Y')

    ax.set_zlim3d([0, 40])
    ax.set_zlabel('Z')

    # Provide starting angle for the view
    elevation_angle = 10
    azimuthal_angle = 45
    ax.view_init(elevation_angle, azimuthal_angle)

    ani = animation.FuncAnimation(fig, animate_scatters, numFrames, fargs=(data, scatters, lines, 
                                                                           scatter_projections, lines_projections,
                                                                           time_text, all_timestamps),
                                       interval=10, blit=False, repeat=True)

    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=100, metadata=dict(artist='LOS'), bitrate=3500)
    ani.save(savePath, writer=writer)
    return

In [31]:
tracks_3D.shape

(501943, 2, 3, 3)

In [33]:
# make the movie

t0 = time.time()

f0 = 0
fE = 100

make_movie(tracks_3D, f0, fE, savePath='/home/liam/temp/testanimation.mp4')

tE = time.time()
print(tE-t0)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

8.836083889007568


# using freshly post-processed data

In [7]:
# Our custom-made imports
import sys
sys.path.append('/home/liam/code/FishTank_Analysis/lib/')
from post_processing import contiguous_regions, process_1D_timeseries

In [8]:
# make lists to hold results
raw_trajectories = []
winner_idxs = []
exp_names = []
exp_numFrames = []
pair_idx = []


trackPath = '/home/liam/fish_analysis/nugget_dump/20201201/FishTank20200130_153857_tracks_A.h5'
with h5py.File(trackPath, 'r') as hf:
    tracks_3D_raw = hf['tracks_3D_raw'][:]

# the fishIdx of the winner fish
winnerIdx = 0
loserIdx = 1
raw_trajectories.append(tracks_3D_raw)
winner_idxs.append(winnerIdx)
exp_numFrames.append(tracks_3D_raw.shape[0])
exp_names.append(os.path.basename(trackPath).split('_')[0] + '_' + os.path.basename(trackPath).split('_')[1])
pair_idx.append(9)


# parse some shapes
numFrames, numFish, numBodyPoints, _ = tracks_3D_raw.shape
fps = 100

# post-process the trajectory
interp_max_gap = 7
interp_polyOrd = 1

smooth_polyOrd = 2
smooth_winSize = 11

tracks_3D_smooth = np.ones_like(tracks_3D_raw)*np.NaN
for fishIdx in range(numFish):
    for bpIdx in range(numBodyPoints):
        for dimIdx in range(3):
            tracks_3D_smooth[:,fishIdx,bpIdx,dimIdx] = process_1D_timeseries(tracks_3D_raw[:,fishIdx,bpIdx,dimIdx],
                                                                             interp_max_gap=interp_max_gap,
                                                                             interp_polyOrd=interp_polyOrd,
                                                                             smooth_polyOrd=smooth_polyOrd,
                                                                             smooth_winSize=smooth_winSize)
            
            
print(tracks_3D_smooth.shape)

(501943, 2, 3, 3)


In [26]:
# make the movie

t0 = time.time()



f0 = 466452
fE = 466761


savePath='/home/liam/temp/loser_pitch_2.mp4'
make_movie(tracks_3D_smooth, f0, fE, savePath=savePath)

tE = time.time()
print(tE-t0)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

26.860577821731567


In [27]:
Video.from_file(savePath)

Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free\x00\x05\xe7\xccmdat\x0…

In [25]:
savePath='/home/liam/temp/loser_pitch_1.mp4'
Video.from_file(savePath)

Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free\x00`\xadPmdat\x00\x00\…

In [24]:
424/60

7.066666666666666

In [9]:

f0 = 465567 
fE = 470557
fE-f0

4990

In [11]:
# empircal from above
movie_writing_speed_fps = 100/8.3
movie_writing_speed_fps

12.048192771084336

In [13]:
movie_writing_speed_fps*(fE-f0)/60

1002.008032128514